# Stability Region Explorer

Chapter 20: ODE, SDE, and Continuous Limits of Algorithms · *Mathematics for AI and Machine Learning* by Fuheng Wu

See when discrete step size destabilizes a continuous stable mode.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-20/stability-region/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: dx/dt = lam x, lam = -1, step h = 0.6, x0 = 1
lam, h, x0 = -1.0, 0.6, 1.0
z = h * lam
phi = 1 + z
print(f"z = h lam = {z}, explicit Euler factor |1 + z| = {abs(phi)} -> {'stable' if abs(phi) < 1 else 'unstable'}")
print("first steps:", [round(x0 * phi ** k, 4) for k in range(8)])

# ── Explicit Euler is stable iff |1 + h lam| < 1: a disk centred at -1 ──────
print("\n   h     |1 + h lam|   x after 50 steps")
for hh in (0.6, 1.5, 1.99, 2.0, 2.5):
    f = 1 + hh * lam
    print(f"{hh:5.2f}     {abs(f):.3f}       {x0 * f ** 50:+.3e}")
print("largest stable step: h < 2 / |lam| =", 2 / abs(lam))

# ── Stiffness: one fast mode forces a tiny explicit step ────────────────────
A = torch.diag(torch.tensor([-1.0, -100.0], dtype=torch.float64))
x = torch.ones(2, dtype=torch.float64)
I = torch.eye(2, dtype=torch.float64)
for hh in (0.01, 0.03):
    xe = torch.linalg.matrix_power(I + hh * A, 100) @ x
    xi = torch.linalg.matrix_power(torch.linalg.inv(I - hh * A), 100) @ x
    print(f"\nstiff system, h = {hh}: explicit -> {[f'{v:.2e}' for v in xe.tolist()]}, implicit -> {[f'{v:.2e}' for v in xi.tolist()]}")
# h = 0.03 > 2/100 blows up explicitly; implicit Euler is stable for every h > 0 when Re(lam) < 0.

# ── Complex lam: an oscillation, stable only inside the disk ────────────────
lam_c = complex(-0.2, 2.0)
for hh in (0.05, 0.3):
    print(f"lam = {lam_c}, h = {hh}: |1 + h lam| = {abs(1 + hh * lam_c):.4f}")

z = h lam = -0.6, explicit Euler factor |1 + z| = 0.4 -> stable
first steps: [1.0, 0.4, 0.16, 0.064, 0.0256, 0.0102, 0.0041, 0.0016]

   h     |1 + h lam|   x after 50 steps
 0.60     0.400       +1.268e-20
 1.50     0.500       +8.882e-16
 1.99     0.990       +6.050e-01
 2.00     1.000       +1.000e+00
 2.50     1.500       +6.376e+08
largest stable step: h < 2 / |lam| = 2.0

stiff system, h = 0.01: explicit -> ['3.66e-01', '0.00e+00'], implicit -> ['3.70e-01', '7.89e-31']

stiff system, h = 0.03: explicit -> ['4.76e-02', '1.27e+30'], implicit -> ['5.20e-02', '6.22e-61']
lam = (-0.2+2j), h = 0.05: |1 + h lam| = 0.9950
lam = (-0.2+2j), h = 0.3: |1 + h lam| = 1.1152
